# Pre-entrega: Análisis de Datos
**Autor:** Franco Barreto
**Fecha:** 04/10/2026
**Entorno:** Python 3.14.4 | Pandas 3.0.5 

---
### Descripción del Proyecto
Este proyecto aborda la primera etapa del ciclo de vida del análisis de datos sobre tres fuentes del negocio:
* `clientes.csv`: Datos demográficos e ingresos estimados de clientes.
* `marketing.csv`: Costos, canales y temporalidad de campañas publicitarias.
* `ventas.csv`: Registro transaccional de compras realizadas.

---

## Etapa 1: Recopilación y Preparación de Datos

### Objetivos:
1. Conexión y carga de datos en DataFrames.
2. Fundamentos de Python: variables, operadores y lógica de cálculo de ventas.
3. Modelado y estructuras de datos nativas (análisis comparativo: Listas vs. Diccionarios).
4. Análisis Exploratorio de Datos (EDA) inicial con Pandas.
5. Auditoría de calidad de datos: detección de nulos, duplicados y reporte de estado.

## Actividad 1: Carga del Set de Datos
En esta sección se inicializa el entorno de trabajo importando las librerías necesarias y cargando el conjunto de datos en un `DataFrame` de Pandas.

> **Nota de reproducibilidad:** Definimos la ruta de los datos mediante una variable para que pueda ser leída tanto desde una URL pública como desde una carpeta local.

In [1]:
import os
import pandas as pd
import numpy as np

# Configuración de visualización para ver todas las columnas
pd.set_option('display.max_columns', None)

# Definición de rutas relativas basadas en la estructura local
RUTA_CARPETA = "files"

try:
    df_clientes = pd.read_csv(os.path.join(RUTA_CARPETA, "clientes.csv"))
    df_marketing = pd.read_csv(os.path.join(RUTA_CARPETA, "marketing.csv"))
    df_ventas = pd.read_csv(os.path.join(RUTA_CARPETA, "ventas.csv"))
    
    print(" Todos los conjuntos de datos fueron cargados con éxito:")
    print(f"• df_clientes  : {df_clientes.shape[0]} filas x {df_clientes.shape[1]} columnas")
    print(f"• df_marketing : {df_marketing.shape[0]} filas x {df_marketing.shape[1]} columnas")
    print(f"• df_ventas    : {df_ventas.shape[0]} filas x {df_ventas.shape[1]} columnas")

except FileNotFoundError as err:
    print(f"❌ Error de ruta: Verifica que los archivos existan en la carpeta '{RUTA_CARPETA}/'. Detalle: {err}")

# Vista previa de control
print("\n--- Muestra: Clientes ---")
display(df_clientes.head(2))

print("\n--- Muestra: Marketing ---")
display(df_marketing.head(2))

print("\n--- Muestra: Ventas ---")
display(df_ventas.head(2))

 Todos los conjuntos de datos fueron cargados con éxito:
• df_clientes  : 567 filas x 5 columnas
• df_marketing : 90 filas x 6 columnas
• df_ventas    : 3035 filas x 6 columnas

--- Muestra: Clientes ---


,id_cliente,nombre,edad,ciudad,ingresos
0,1,Aloysia Screase,44,Mar del Plata,42294.68
1,2,Kristina Scaplehorn,25,Posadas,24735.04



--- Muestra: Marketing ---


,id_campanha,producto,canal,costo,fecha_inicio,fecha_fin
0,74,Adorno de pared,TV,4.81,20/03/2024,03/05/2024
1,12,Tablet,RRSS,3.40,26/03/2024,13/05/2024



--- Muestra: Ventas ---


,id_venta,producto,precio,cantidad,fecha_venta,categoria
0,792,Cuadro decorativo,$69.94,5.0,02/01/2024,Decoración
1,811,Lámpara de mesa,$105.10,5.0,02/01/2024,Decoración


## Actividad 2: Cálculo Dinámico de Ventas Mensuales (Todos los Meses del Año)
En esta sección se procesa el dataset completo `files/ventas.csv` utilizando Python nativo (`open()` y `csv.DictReader`). 

El script agrupa las transacciones por período mensual (`MM/YYYY`), aplica operadores aritméticos (`*`, `+=`, `/`) para calcular la recaudación total, unidades vendidas y ticket promedio de **cada uno de los 12 meses**, y finalmente determina el **promedio mensual general** del negocio.

In [3]:
import csv
import os

# 1. Función auxiliar para dar formato de moneda/número en Argentina (1.234,56)
def formato_ar(valor, es_moneda=True, decimales=2):
    """
    Convierte números al estándar argentino: punto para miles y coma para decimales.
    Ejemplo: 129604.99 -> '$ 129.604,99' o 1741 -> '1.741'
    """
    if decimales > 0:
        texto = f"{valor:,.{decimales}f}"
    else:
        texto = f"{int(valor):,}"
    
    # Intercambiamos comas por puntos (miles) y punto por coma (decimal)
    texto_ar = texto.replace(",", "X").replace(".", ",").replace("X", ".")
    
    return f"$ {texto_ar}" if es_moneda else texto_ar

# 2. Diccionario ordenado para mapear el código del mes con su nombre en español
MESES_NOMBRE = {
    "01": "Enero",      "02": "Febrero",    "03": "Marzo", 
    "04": "Abril",      "05": "Mayo",       "06": "Junio", 
    "07": "Julio",      "08": "Agosto",     "09": "Septiembre", 
    "10": "Octubre",    "11": "Noviembre",  "12": "Diciembre"
}

# 3. Ruta al archivo de ventas
ruta_ventas = os.path.join("files", "ventas.csv")

# 4. Inicializamos el diccionario de meses con ceros para asegurar orden cronológico
ventas_por_mes = {
    codigo: {"recaudacion": 0.0, "unidades": 0, "operaciones": 0} 
    for codigo in MESES_NOMBRE.keys()
}

# 5. Lectura y procesamiento del archivo con Python nativo
with open(ruta_ventas, mode="r", encoding="utf-8") as archivo:
    lector = csv.DictReader(archivo)
    
    for fila in lector:
        fecha = fila["fecha_venta"]
        precio_raw = fila["precio"]
        cantidad_raw = fila["cantidad"]
        
        # Filtro de sanidad: descartar filas con campos vacíos
        if precio_raw and cantidad_raw and fecha:
            partes_fecha = fecha.split("/")
            if len(partes_fecha) == 3:
                num_mes = partes_fecha[1]  # Extraemos '01', '02', etc.
                
                if num_mes in ventas_por_mes:
                    precio = float(precio_raw.replace("$", "").strip())
                    cantidad = int(cantidad_raw)
                    
                    # Operador aritmético de multiplicación (*)
                    subtotal = precio * cantidad
                    
                    # Operadores aritméticos de acumulación (+=)
                    ventas_por_mes[num_mes]["recaudacion"] += subtotal
                    ventas_por_mes[num_mes]["unidades"] += cantidad
                    ventas_por_mes[num_mes]["operaciones"] += 1

# 6. Variables globales para los totales del año
recaudacion_anual = 0.0
unidades_anual = 0
operaciones_anual = 0

print("=" * 72)
print("             REPORTE CONSOLIDADO DE VENTAS MENSUALES (2024)")
print("=" * 72)
print(f"{'MES':<12} | {'TRANSAC.':>9} | {'UNIDADES':>9} | {'RECAUDACIÓN':>18} | {'TICKET PROM.':>14}")
print("-" * 72)

# Iteramos en orden cronológico del '01' al '12'
for codigo, nombre_mes in MESES_NOMBRE.items():
    datos = ventas_por_mes[codigo]
    rec = datos["recaudacion"]
    uni = datos["unidades"]
    ops = datos["operaciones"]
    
    # Operador de división (/) para el ticket promedio
    ticket_prom = rec / ops if ops > 0 else 0.0
    
    # Acumulamos en las variables anuales
    recaudacion_anual += rec
    unidades_anual += uni
    operaciones_anual += ops
    
    # Imprimimos la fila con formato regional argentino
    print(f"{nombre_mes:<12} | {ops:>9} | {formato_ar(uni, es_moneda=False, decimales=0):>9} | {formato_ar(rec):>18} | {formato_ar(ticket_prom):>14}")

print("-" * 72)

# 7. Promedios mensuales generales
cantidad_meses = len(MESES_NOMBRE)
promedio_recaudacion_mensual = recaudacion_anual / cantidad_meses
promedio_unidades_mensual = unidades_anual / cantidad_meses

print(f"\n RESUMEN GLOBAL (AÑO 2024):")
print(f"• Total Facturado en el año     : {formato_ar(recaudacion_anual)}")
print(f"• Total Unidades vendidas       : {formato_ar(unidades_anual, es_moneda=False, decimales=0)} unidades")
print(f"• Meses evaluados               : {cantidad_meses} meses")
print(f"• Promedio facturado por mes    : {formato_ar(promedio_recaudacion_mensual)} / mes")
print(f"• Promedio de unidades por mes  : {formato_ar(promedio_unidades_mensual, es_moneda=False, decimales=1)} unidades / mes")
print("=" * 72)

             REPORTE CONSOLIDADO DE VENTAS MENSUALES (2024)
MES          |  TRANSAC. |  UNIDADES |        RECAUDACIÓN |   TICKET PROM.
------------------------------------------------------------------------
Enero        |       268 |     1.741 |       $ 129.604,99 |       $ 483,60
Febrero      |       248 |     1.607 |       $ 118.672,44 |       $ 478,52
Marzo        |       265 |     1.792 |       $ 136.779,15 |       $ 516,15
Abril        |       310 |     1.924 |       $ 144.380,10 |       $ 465,74
Mayo         |       275 |     1.812 |       $ 143.727,25 |       $ 522,64
Junio        |       221 |     1.426 |       $ 108.480,17 |       $ 490,86
Julio        |       243 |     1.547 |       $ 116.229,97 |       $ 478,31
Agosto       |       254 |     1.614 |       $ 119.680,15 |       $ 471,18
Septiembre   |       241 |     1.576 |       $ 115.787,85 |       $ 480,45
Octubre      |       230 |     1.458 |       $ 112.117,13 |       $ 487,47
Noviembre    |       241 |     1.625 |    

## Actividad 3: Estructuras de Datos Nativas

### Consigna:
> Desarrollar un programa que almacene los datos de ventas (producto, precio, cantidad). Decidir si conviene utilizar diccionarios o listas.

### Decisión de Ingeniería de Datos:
Se optó por una **Lista de Diccionarios (`list[dict]`)**:
1. **Semántica y Robustez:** Al usar diccionarios internos, los atributos se consultan por clave (`item['precio']`) en lugar de posición numérica (`item[1]`), evitando fallos si el orden del archivo cambia.
2. **Cardinalidad:** Una lista permite contener los más de 1.000 registros transaccionales sin riesgo de sobreescritura de claves repetidas.
3. **Interoperabilidad:** Es la representación nativa estándar (JSON) antes de migrar a estructuras matriciales como los DataFrames de Pandas.

In [5]:
import csv
import os

# =====================================================================
# PROGRAMA: GESTOR DE ALMACENAMIENTO DE VENTAS
# Estructura elegida: Lista de Diccionarios (list[dict])
# =====================================================================

class AlmacenVentas:
    """
    Programa para gestionar el almacenamiento en memoria de transacciones comerciales.
    Se utiliza una Lista de Diccionarios para garantizar:
    - Búsqueda y acceso semántico por clave (item['precio']).
    - Capacidad de albergar múltiples transacciones sin colisión de nombres.
    """
    def __init__(self):
        # Estructura principal en memoria
        self.registro_ventas = []

    def almacenar_venta(self, producto: str, precio: float, cantidad: int) -> bool:
        """Valida y almacena una nueva venta en la estructura elegida."""
        if not producto or precio <= 0 or cantidad <= 0:
            print(f"❌ Error: Datos inválidos para el producto '{producto}'.")
            return False

        # Almacenamiento en diccionario estructurado
        venta = {
            "producto": str(producto).strip(),
            "precio": float(precio),
            "cantidad": int(cantidad)
        }
        self.registro_ventas.append(venta)
        return True

    def cargar_desde_csv(self, ruta_archivo: str):
        """Puebla el almacenamiento con los datos transaccionales existentes."""
        if not os.path.exists(ruta_archivo):
            print(f"❌ Archivo no encontrado: {ruta_archivo}")
            return

        cargados = 0
        with open(ruta_archivo, mode="r", encoding="utf-8") as archivo:
            lector = csv.DictReader(archivo)
            for fila in lector:
                prod = fila.get("producto")
                p_raw = fila.get("precio")
                c_raw = fila.get("cantidad")

                if prod and p_raw and c_raw:
                    try:
                        p = float(p_raw.replace("$", "").strip())
                        c = int(c_raw)
                        if self.almacenar_venta(prod, p, c):
                            cargados += 1
                    except ValueError:
                        continue  # Omitir filas corruptas

        print(f" Se almacenaron con éxito {cargados:,} transacciones en la estructura de memoria.")

    def calcular_totales(self):
        """Procesa y calcula métricas sobre la estructura almacenada."""
        total_ingresos = sum(item["precio"] * item["cantidad"] for item in self.registro_ventas)
        total_articulos = sum(item["cantidad"] for item in self.registro_ventas)
        return total_ingresos, total_articulos

    def mostrar_resumen(self, limite=5):
        """Muestra una vista previa de las transacciones almacenadas."""
        print(f"\n--- MUESTRA DE DATOS ALMACENADOS (Primeros {limite}) ---")
        for i, item in enumerate(self.registro_ventas[:limite], start=1):
            subtotal = item["precio"] * item["cantidad"]
            print(f"{i}. {item['producto']:<22} | Cant: {item['cantidad']:>2} | Unit: $ {item['precio']:>8.2f} | Subtotal: $ {subtotal:>9.2f}")


# -------------------------------------------------------------
# EJECUCIÓN Y PRUEBA DEL PROGRAMA
# -------------------------------------------------------------

# 1. Instanciamos el programa
gestor = AlmacenVentas()

# 2. Poblar el almacén con el archivo real 'ventas.csv'
gestor.cargar_desde_csv(os.path.join("files", "ventas.csv"))

# 3. Demostración de almacenamiento de NUEVAS ventas ingresadas al programa
print("\n Registrando nuevas ventas manuales en el almacén:")
gestor.almacenar_venta("Smart TV 55 Pulgadas", 450.00, 2)
gestor.almacenar_venta("Silla Ergonómica Pro", 180.50, 4)

# 4. Inspección y totales
gestor.mostrar_resumen(limite=5)

facturacion_total, unidades_totales = gestor.calcular_totales()
print("\n" + "=" * 55)
print(f"Total registros almacenados : {len(gestor.registro_ventas):,} ventas")
print(f"Total unidades acumuladas   : {unidades_totales:,} unidades")
print(f"Facturación total calculada : $ {facturacion_total:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
print("=" * 55)

 Se almacenaron con éxito 3,033 transacciones en la estructura de memoria.

 Registrando nuevas ventas manuales en el almacén:

--- MUESTRA DE DATOS ALMACENADOS (Primeros 5) ---
1. Cuadro decorativo      | Cant:  5 | Unit: $    69.94 | Subtotal: $    349.70
2. Lámpara de mesa        | Cant:  5 | Unit: $   105.10 | Subtotal: $    525.50
3. Secadora               | Cant:  3 | Unit: $    97.96 | Subtotal: $    293.88
4. Heladera               | Cant:  8 | Unit: $   114.35 | Subtotal: $    914.80
5. Secadora               | Cant:  4 | Unit: $   106.21 | Subtotal: $    424.84

Total registros almacenados : 3,035 ventas
Total unidades acumuladas   : 19,710 unidades
Facturación total calculada : $ 1.484.664,93


## Actividad 4: Introducción a Pandas - Análisis Exploratorio Inicial (EDA)
Inspección general de la estructura, dimensionalidad, tipos de datos asignados y estadísticas descriptivas de cada dataset cargado.

In [6]:
def inspeccion_inicial(df, nombre):
    print("=" * 60)
    print(f"   ANÁLISIS EXPLORATORIO: {nombre}")
    print("=" * 60)
    print(f"Forma (Shape): {df.shape[0]} filas, {df.shape[1]} columnas\n")
    print("--- Tipos de Datos y Valores no Nulos (df.info()) ---")
    df.info()
    print("\n--- Estadísticas Descriptivas (df.describe()) ---")
    display(df.describe(include='all'))
    print("\n")

# Ejecutamos la inspección en los tres DataFrames
inspeccion_inicial(df_clientes, "CLIENTES")
inspeccion_inicial(df_marketing, "MARKETING")
inspeccion_inicial(df_ventas, "VENTAS")

   ANÁLISIS EXPLORATORIO: CLIENTES
Forma (Shape): 567 filas, 5 columnas

--- Tipos de Datos y Valores no Nulos (df.info()) ---
<class 'pandas.DataFrame'>
RangeIndex: 567 entries, 0 to 566
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id_cliente  567 non-null    int64  
 1   nombre      567 non-null    str    
 2   edad        567 non-null    int64  
 3   ciudad      567 non-null    str    
 4   ingresos    567 non-null    float64
dtypes: float64(1), int64(2), str(2)
memory usage: 35.6 KB

--- Estadísticas Descriptivas (df.describe()) ---


,id_cliente,nombre,edad,ciudad,ingresos
count,567.000000,567,567.000000,567,567.000000
unique,NaN,567,NaN,12,NaN
top,NaN,Aloysia Screase,NaN,Mar del Plata,NaN
freq,NaN,1,NaN,63,NaN
mean,284.000000,NaN,37.940035,NaN,34668.739012
std,163.823075,NaN,10.202885,NaN,12974.531446
min,1.000000,NaN,20.000000,NaN,170.290000
25%,142.500000,NaN,30.000000,NaN,26015.240000
50%,284.000000,NaN,37.000000,NaN,35066.830000
75%,425.500000,NaN,43.000000,NaN,42457.100000




   ANÁLISIS EXPLORATORIO: MARKETING
Forma (Shape): 90 filas, 6 columnas

--- Tipos de Datos y Valores no Nulos (df.info()) ---
<class 'pandas.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_campanha   90 non-null     int64  
 1   producto      90 non-null     str    
 2   canal         90 non-null     str    
 3   costo         90 non-null     float64
 4   fecha_inicio  90 non-null     str    
 5   fecha_fin     90 non-null     str    
dtypes: float64(1), int64(1), str(4)
memory usage: 7.5 KB

--- Estadísticas Descriptivas (df.describe()) ---


,id_campanha,producto,canal,costo,fecha_inicio,fecha_fin
count,90.000000,90,90,90.000000,90,90
unique,NaN,30,3,NaN,78,78
top,NaN,Adorno de pared,TV,NaN,31/05/2024,05/05/2024
freq,NaN,3,30,NaN,2,3
mean,45.500000,NaN,NaN,4.928667,NaN,NaN
std,26.124701,NaN,NaN,0.947750,NaN,NaN
min,1.000000,NaN,NaN,2.950000,NaN,NaN
25%,23.250000,NaN,NaN,4.372500,NaN,NaN
50%,45.500000,NaN,NaN,4.900000,NaN,NaN
75%,67.750000,NaN,NaN,5.562500,NaN,NaN




   ANÁLISIS EXPLORATORIO: VENTAS
Forma (Shape): 3035 filas, 6 columnas

--- Tipos de Datos y Valores no Nulos (df.info()) ---
<class 'pandas.DataFrame'>
RangeIndex: 3035 entries, 0 to 3034
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_venta     3035 non-null   int64  
 1   producto     3035 non-null   str    
 2   precio       3033 non-null   str    
 3   cantidad     3033 non-null   float64
 4   fecha_venta  3035 non-null   str    
 5   categoria    3035 non-null   str    
dtypes: float64(1), int64(1), str(4)
memory usage: 269.3 KB

--- Estadísticas Descriptivas (df.describe()) ---


,id_venta,producto,precio,cantidad,fecha_venta,categoria
count,3035.000000,3035,3033,3033.000000,3035,3035
unique,NaN,30,2590,NaN,364,3
top,NaN,Lámpara de mesa,$76.32,NaN,06/04/2024,Decoración
freq,NaN,181,5,NaN,24,1015
mean,1499.851400,NaN,NaN,6.496538,NaN,NaN
std,866.465379,NaN,NaN,3.457250,NaN,NaN
min,1.000000,NaN,NaN,1.000000,NaN,NaN
25%,748.500000,NaN,NaN,3.000000,NaN,NaN
50%,1502.000000,NaN,NaN,7.000000,NaN,NaN
75%,2249.500000,NaN,NaN,9.000000,NaN,NaN


### Hallazgos del Análisis Exploratorio de Datos (EDA)

A partir de la inspección técnica (`info()`) y estadística (`describe()`) de los tres DataFrames, se identifican los siguientes patrones, métricas de negocio y anomalías técnicas:

---

#### 1. Dataset de Clientes (`df_clientes` - 567 filas × 5 columnas)
* **Demografía y Cobertura:** 
  * Los clientes residen en **12 ciudades distintas**, siendo **Mar del Plata la principal plaza** con 63 clientes registrados (~11% del total).
  * La edad oscila entre los **20 y los 81 años**, con un promedio de **~38 años** (el 50% de los clientes tiene 37 años o menos).
* **Anomalía en Ingresos (Alerta de Outlier):**
  * Si bien la media de ingresos es de **$34.668,74** y el máximo alcanza los **$88.053,01**, el valor mínimo registrado es de apenas **$170,29**. 
  * *Diagnóstico:* Este valor atípicamente bajo sugiere un error de tipeo o un caso extremo que deberá auditarse en la fase de calidad.
* **Integridad:** No presenta valores nulos en ninguna de sus 5 columnas y los 567 nombres son completamente únicos.

---

#### 2. Dataset de Marketing (`df_marketing` - 90 filas × 6 columnas)
* **Distribución de Canales:** 
  * Se registran exactamente 90 campañas divididas en **3 canales publicitarios (TV, RRSS y Email)**, con una asignación perfectamente equitativa de **30 campañas por canal**.
* **Presupuesto y Alcance:**
  * Se promocionan **30 productos únicos** (coincide exactamente con la oferta de productos en ventas).
  * El costo de campaña promedio es de **$4,93** (mínimo de $2,95 y máximo de $7,39).
* **Detalle Técnico:** Las columnas `fecha_inicio` y `fecha_fin` están almacenadas como texto (`str`/`object`) en formato `DD/MM/YYYY`, lo que impedirá calcular la duración de las campañas hasta que se conviertan a `datetime`.

---

#### 3. Dataset de Ventas (`df_ventas` - 3.035 filas × 6 columnas)
* **Discrepancia en Identificadores (Evidencia de Duplicados):**
  * La tabla cuenta con **3.035 filas**, pero el valor máximo de `id_venta` es **3.000**. Esto demuestra matemáticamente que existen identificadores repetidos y registros duplicados.
* **Presencia de Valores Faltantes (NaNs):**
  * `precio` y `cantidad` tienen solo **3.033 registros no nulos** (faltan exactamente 2 valores en cada columna).
  * *Efecto colateral:* Debido a esos 2 nulos, Pandas forzó la columna `cantidad` a tipo `float64` (decimal) en lugar de entero (`int64`).
* **Tratamiento de Precios:** La columna `precio` se infirió como texto (`str`) por la presencia del caracter `$`, requiriendo sanitización para operar matemáticamente.
* **Comportamiento Comercial:**
  * Las transacciones abarcan **3 categorías**, lideradas por **Decoración** con 1.015 ventas.
  * El catálogo activo es de **30 productos**, siendo la **Lámpara de mesa** el artículo más vendido en frecuencia (181 transacciones).
  * El volumen por ticket varía entre **1 y 12 unidades**, con un promedio de **~6,5 unidades por compra**.
  * Se registran ventas en **364 días del año 2024**, teniendo el pico diario el **06/04/2024** con 24 operaciones.

## Actividad 5: Calidad de Datos (Nulos y Duplicados)
Identificación y diagnóstico del estado de sanidad de los datos iniciales, **sin modificar ni eliminar registros todavía** (siguiendo la pauta de auditoría inicial).

In [8]:
def auditar_calidad(df, nombre):
    print("=" * 55)
    print(f" AUDITORÍA DE CALIDAD: {nombre}")
    print("=" * 55)
    
    # 1. Nulos
    nulos_conteo = df.isnull().sum()
    nulos_pct = (df.isnull().sum() / len(df)) * 100
    tabla_nulos = pd.DataFrame({'Nulos': nulos_conteo, 'Porcentaje (%)': nulos_pct.round(2)})
    
    hay_nulos = nulos_conteo.sum() > 0
    if hay_nulos:
        print(" Columnas con valores nulos detectadas:")
        display(tabla_nulos[tabla_nulos['Nulos'] > 0])
    else:
        print(" Cero valores nulos detectados.")
        
    # 2. Duplicados
    duplicados_totales = df.duplicated().sum()
    duplicados_pct = (duplicados_totales / len(df)) * 100
    print(f"\n Registros duplicados exactos: {duplicados_totales} ({duplicados_pct:.2f}%)")
    
    if duplicados_totales > 0:
        print("Muestra de filas duplicadas:")
        display(df[df.duplicated(keep=False)].head(6))
    print("\n")

# Ejecutar auditoría sobre los tres datasets
auditar_calidad(df_clientes, "df_clientes")
auditar_calidad(df_marketing, "df_marketing")
auditar_calidad(df_ventas, "df_ventas")

 AUDITORÍA DE CALIDAD: df_clientes
 Cero valores nulos detectados.

 Registros duplicados exactos: 0 (0.00%)


 AUDITORÍA DE CALIDAD: df_marketing
 Cero valores nulos detectados.

 Registros duplicados exactos: 0 (0.00%)


 AUDITORÍA DE CALIDAD: df_ventas
 Columnas con valores nulos detectadas:


,Nulos,Porcentaje (%)
precio,2,0.07
cantidad,2,0.07



 Registros duplicados exactos: 35 (1.15%)
Muestra de filas duplicadas:


,id_venta,producto,precio,cantidad,fecha_venta,categoria
820,56,Cortinas,$66.24,5.0,05/04/2024,Decoración
821,421,Lámpara de mesa,$114.83,9.0,05/04/2024,Decoración
822,424,Jarrón decorativo,$87.94,2.0,05/04/2024,Decoración
823,1868,Cafetera,$62.23,1.0,05/04/2024,Electrodomésticos
824,2545,Auriculares,$32.81,11.0,05/04/2024,Electrónica
825,2778,SmartWatch,$70.66,7.0,05/04/2024,Electrónica


### Informe Ejecutivo de Auditoría de Calidad de Datos (Estado Inicial)

Habiendo ejecutado la auditoría algorítmica sobre los tres conjuntos de datos del negocio (`clientes`, `marketing` y `ventas`), se documenta el estado inicial de sanidad **sin haber aplicado alteraciones ni eliminación de registros**:

---

#### 1. Matriz de Control y Diagnóstico Inicial

| Conjunto de Datos | Registros Totales | Columnas | Valores Nulos (Missing Values) | Registros Duplicados Exactos | Estado de Sanidad |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **`df_clientes`** | **567** | 5 | **0** (0,00%) | **0** (0,00%) |  **Óptimo / Confiable** |
| **`df_marketing`** | **90** | 6 | **0** (0,00%) | **0** (0,00%) |  **Estructuralmente Sano** |
| **`df_ventas`** | **3.035** | 6 | **4** (2 en `precio`, 2 en `cantidad`) | **35** (1,15%) | ⚠️ **Requiere Limpieza Crítica** |

---

#### 2. Hallazgos Específicos por Entidad

##### A. Clientes (`clientes.csv`):
* **Integridad estructural:** 100% de celdas completas, sin identificadores repetidos ni registros duplicados.
* **Punto de atención analítico:** Se detectó un valor de ingreso mínimo de **\$ 170,29** extremadamente distante de la media ($ 34.668,74). No representa un nulo técnico, pero constituye un **posible valor atípico (outlier)** o error de entrada humana a monitorear.

##### B. Marketing (`marketing.csv`):
* **Integridad estructural:** Sin nulos ni duplicados. Cobertura simétrica de 30 campañas por canal (TV, RRSS, Email).
* **Deuda técnica de formato:** Las columnas `fecha_inicio` y `fecha_fin` se leyeron como tipo texto (`str`).

##### C. Ventas (`ventas.csv`) - *Foco de Intervención para la Etapa 2*:
1. **Registros Duplicados:** Se confirmaron **35 filas idénticas** (1,15% del dataset). Al eliminar estos duplicados, el conjunto de datos convergerá exactamente a las 3.000 transacciones únicas originales.
2. **Valores Faltantes:** Se identificaron **2 registros corruptos** donde tanto el `precio` como la `cantidad` son nulos de forma simultánea (específicamente en los IDs 627 y 2171, correspondientes al 0,07% del volumen).
3. **Casting y Tipos de Datos Forzados:**
   * La presencia del símbolo `$` forzó a la columna `precio` a ser importada como texto (`str`), impidiendo cálculos analíticos directos con Pandas.
   * La presencia de los 2 valores nulos en `cantidad` obligó a Pandas a convertir la columna a decimales (`float64`) en lugar de números enteros (`int64`).
   * La columna `fecha_venta` requiere ser casteada a `datetime`.

---

#### 3. Plan de Acción Recomendado para la Etapa 2 (Limpieza y Transformación)

1. **Deduplicación:** Aplicar `df_ventas.drop_duplicates(inplace=True)` para remover las 35 transacciones redundantes.
2. **Tratamiento de Nulos:** Dado que los nulos en `df_ventas` representan apenas el 0,07% y carecen simultáneamente de precio y cantidad, la estrategia recomendada es la **eliminación de esas 2 filas** (`dropna()`), ya que no es posible imputar con certeza una transacción comercial sin monto ni unidades.
3. **Normalización de Tipos:**
   * Sanitizar `precio`: `df_ventas['precio'] = df_ventas['precio'].str.replace('$', '').astype(float)`
   * Re-castear `cantidad` a entero: `df_ventas['cantidad'] = df_ventas['cantidad'].astype(int)`
   * Convertir fechas a estándar temporal: `pd.to_datetime()` en `ventas` y `marketing`.

---
# Etapa 2: Preprocesamiento y Limpieza de Datos (Clases 5 a 8)
**Objetivo:** Aplicar técnicas profesionales de limpieza, transformación, agregación e integración sobre los conjuntos de datos auditados en la Etapa 1.

## Actividad 1: Limpieza de Datos (Duplicados, Caracteres y Normalización)

### Diagnóstico previo aplicado:
1. **`df_ventas`:**
   * Eliminación de las **35 filas duplicadas exactas**.
   * Sanitización del campo `precio`: eliminación del caracter `$` y conversión a decimal (`float64`).
   * Remoción de los **2 registros nulos** (sin precio ni cantidad en las transacciones 627 y 2171).
   * Casteo de `cantidad` a entero (`int64`) y creación de la columna calculada `ingreso_total = precio * cantidad`.
   * Conversión de `fecha_venta` a formato temporal `datetime`.
2. **`df_marketing`:**
   * Conversión de `fecha_inicio` y `fecha_fin` a formato temporal `datetime`.

In [9]:
# =====================================================================
# 1. LIMPIEZA DE DF_VENTAS
# =====================================================================

# Creamos una copia explícita para preservar el dataset crudo
df_ventas_limpio = df_ventas.copy()

filas_iniciales_ventas = len(df_ventas_limpio)

# a) Eliminación de duplicados exactos
df_ventas_limpio.drop_duplicates(inplace=True)
duplicados_removidos = filas_iniciales_ventas - len(df_ventas_limpio)

# b) Tratamiento de nulos (dropna en precio y cantidad)
df_ventas_limpio.dropna(subset=['precio', 'cantidad'], inplace=True)
nulos_removidos = (filas_iniciales_ventas - duplicados_removidos) - len(df_ventas_limpio)

# c) Eliminación del caracter no deseado '$' en precio y conversión a float
df_ventas_limpio['precio'] = (
    df_ventas_limpio['precio']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
    .astype(float)
)

# d) Casteo de cantidad a entero
df_ventas_limpio['cantidad'] = df_ventas_limpio['cantidad'].astype(int)

# e) Creación de columna calculada de ingresos por transacción
df_ventas_limpio['ingreso_total'] = df_ventas_limpio['precio'] * df_ventas_limpio['cantidad']

# f) Conversión de fecha a datetime
df_ventas_limpio['fecha_venta'] = pd.to_datetime(df_ventas_limpio['fecha_venta'], format='%d/%m/%Y')

# =====================================================================
# 2. LIMPIEZA DE DF_MARKETING
# =====================================================================

df_marketing_limpio = df_marketing.copy()

# Conversión de fechas de inicio y fin de campaña
df_marketing_limpio['fecha_inicio'] = pd.to_datetime(df_marketing_limpio['fecha_inicio'], format='%d/%m/%Y')
df_marketing_limpio['fecha_fin'] = pd.to_datetime(df_marketing_limpio['fecha_fin'], format='%d/%m/%Y')

# Columna derivada: Duración de la campaña publicitaria en días
df_marketing_limpio['duracion_dias'] = (df_marketing_limpio['fecha_fin'] - df_marketing_limpio['fecha_inicio']).dt.days

# =====================================================================
# REPORTE DE RESULTADOS DE LIMPIEZA
# =====================================================================

print("=" * 60)
print("        REPORTE DE TRANSFORMACIÓN Y LIMPIEZA")
print("=" * 60)
print(f"• df_ventas original        : {filas_iniciales_ventas:,} filas")
print(f"• Duplicados eliminados     : {duplicados_removidos:,} filas")
print(f"• Registros nulos removidos : {nulos_removidos:,} filas")
print(f"• df_ventas final limpio    : {len(df_ventas_limpio):,} filas limpias")
print("-" * 60)
print("Tipos de datos actualizados en df_ventas_limpio:")
print(df_ventas_limpio.dtypes)
print("=" * 60)

display(df_ventas_limpio.head(3))

        REPORTE DE TRANSFORMACIÓN Y LIMPIEZA
• df_ventas original        : 3,035 filas
• Duplicados eliminados     : 35 filas
• Registros nulos removidos : 2 filas
• df_ventas final limpio    : 2,998 filas limpias
------------------------------------------------------------
Tipos de datos actualizados en df_ventas_limpio:
id_venta                  int64
producto                    str
precio                  float64
cantidad                  int64
fecha_venta      datetime64[us]
categoria                   str
ingreso_total           float64
dtype: object


,id_venta,producto,precio,cantidad,fecha_venta,categoria,ingreso_total
0,792,Cuadro decorativo,69.94,5,2024-01-02,Decoración,349.70
1,811,Lámpara de mesa,105.10,5,2024-01-02,Decoración,525.50
2,1156,Secadora,97.96,3,2024-01-02,Electrodomésticos,293.88


## Actividad 2: Transformación de Datos (Productos de Alto Rendimiento)

### Criterio de Selección:
Para identificar objetivamente a los **productos de alto rendimiento**, se calcula la facturación total acumulada por producto. 

Se define como producto de alto rendimiento a aquel cuyos ingresos superan el **promedio de facturación global por producto** del catálogo:
$$\text{Ingreso Producto} > \text{Ingreso Promedio Catálogo}$$

In [10]:
# 1. Agrupación por producto para medir rendimiento comercial
rendimiento_productos = (
    df_ventas_limpio
    .groupby('producto')
    .agg(
        ingreso_acumulado=('ingreso_total', 'sum'),
        unidades_vendidas=('cantidad', 'sum'),
        transacciones=('id_venta', 'count'),
        precio_promedio=('precio', 'mean')
    )
    .reset_index()
)

# 2. Umbral de corte: Ingreso promedio del catálogo
umbral_facturacion = rendimiento_productos['ingreso_acumulado'].mean()

# 3. Filtrado: Solo productos con facturación superior al promedio
productos_alto_rendimiento = (
    rendimiento_productos[rendimiento_productos['ingreso_acumulado'] > umbral_facturacion]
    .sort_values(by='ingreso_acumulado', ascending=False)
    .reset_index(drop=True)
)

# 4. Presentación con formato de moneda
tabla_alto_rendimiento_vista = productos_alto_rendimiento.copy()
tabla_alto_rendimiento_vista['ingreso_acumulado'] = tabla_alto_rendimiento_vista['ingreso_acumulado'].apply(
    lambda x: f"$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
tabla_alto_rendimiento_vista['precio_promedio'] = tabla_alto_rendimiento_vista['precio_promedio'].apply(
    lambda x: f"$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
tabla_alto_rendimiento_vista['unidades_vendidas'] = tabla_alto_rendimiento_vista['unidades_vendidas'].apply(
    lambda x: f"{x:,}".replace(",", ".")
)

print(f" Umbral de corte (Facturación promedio por producto): $ {umbral_facturacion:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
print(f" De {len(rendimiento_productos)} productos en el catálogo, {len(productos_alto_rendimiento)} califican como 'Alto Rendimiento':\n")

display(tabla_alto_rendimiento_vista)

 Umbral de corte (Facturación promedio por producto): $ 48.903,12
 De 30 productos en el catálogo, 14 califican como 'Alto Rendimiento':



,producto,ingreso_acumulado,unidades_vendidas,transacciones,precio_promedio
0,Lámpara de mesa,"$ 82.276,38",1.112,176,"$ 72,72"
1,Auriculares,"$ 74.175,58",958,143,"$ 76,30"
2,Microondas,"$ 72.562,89",912,135,"$ 79,18"
3,Cafetera,"$ 59.607,31",765,117,"$ 79,05"
4,Cuadro decorativo,"$ 54.297,60",726,100,"$ 74,58"
5,Smartphone,"$ 54.132,44",665,101,"$ 81,40"
6,Secadora,"$ 52.115,45",696,100,"$ 73,34"
7,Jarrón decorativo,"$ 51.130,88",672,100,"$ 74,53"
8,Batidora,"$ 50.979,20",672,100,"$ 77,54"
9,Rincón de plantas,"$ 50.456,45",691,101,"$ 74,82"


## Actividad 3: Agregación (Resumen de Ventas por Categoría)
Se resume el comportamiento de las ventas agrupadas por `categoria` para responder:
* ¿Qué categoría genera el mayor volumen de facturación?
* ¿Cuál tiene el ticket promedio más alto?
* ¿Cómo se distribuye porcentualmente el aporte de cada una a los ingresos totales?

In [11]:
# 1. Agregación por categoría
resumen_categoria = (
    df_ventas_limpio
    .groupby('categoria')
    .agg(
        ingresos_totales=('ingreso_total', 'sum'),
        unidades_totales=('cantidad', 'sum'),
        operaciones=('id_venta', 'count'),
        ticket_promedio=('ingreso_total', 'mean')
    )
    .reset_index()
)

# 2. Participación porcentual sobre la facturación global
total_global_ventas = resumen_categoria['ingresos_totales'].sum()
resumen_categoria['participacion_pct'] = (resumen_categoria['ingresos_totales'] / total_global_ventas) * 100

# Ordenamos por recaudación descendente
resumen_categoria.sort_values(by='ingresos_totales', ascending=False, inplace=True)

# 3. Vista con formato regional
vista_categoria = resumen_categoria.copy()
vista_categoria['ingresos_totales'] = vista_categoria['ingresos_totales'].apply(
    lambda x: f"$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
vista_categoria['ticket_promedio'] = vista_categoria['ticket_promedio'].apply(
    lambda x: f"$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
vista_categoria['unidades_totales'] = vista_categoria['unidades_totales'].apply(
    lambda x: f"{x:,}".replace(",", ".")
)
vista_categoria['participacion_pct'] = vista_categoria['participacion_pct'].apply(
    lambda x: f"{x:.2f} %".replace(".", ",")
)

print("=" * 75)
print("             RESUMEN EJECUTIVO DE VENTAS POR CATEGORÍA")
print("=" * 75)
display(vista_categoria)

             RESUMEN EJECUTIVO DE VENTAS POR CATEGORÍA


,categoria,ingresos_totales,unidades_totales,operaciones,ticket_promedio,participacion_pct
1,Electrodomésticos,"$ 505.299,63",6.592,1000,"$ 505,30","34,44 %"
2,Electrónica,"$ 482.577,80",6.413,998,"$ 483,54","32,89 %"
0,Decoración,"$ 479.216,09",6.490,1000,"$ 479,22","32,66 %"


### Conclusiones del Análisis por Categoría:
1. **Liderazgo en Ingresos:** La categoría con mayor facturación es analizada en la tabla anterior, superando el 30% del volumen comercial total.
2. **Volumen de transacciones:** Decoración, Electrodomésticos y Electrónica muestran una distribución competitiva, pero con diferencias marcadas en el ticket promedio por operación.

## Actividad 4: Integración de Datos (Ventas + Marketing)

### Hipótesis de Negocio e Integración:
Tanto `df_ventas_limpio` como `df_marketing_limpio` comparten el atributo **`producto`**. 

Al integrar ambos datasets mediante un **Merge Interno (`inner join`)**, podemos cruzar:
* **Inversión publicitaria total** por producto y los canales utilizados (`costo`).
* **Ingresos reales generados** por cada producto (`ingreso_total`).
* **Retorno sobre la Inversión en Marketing (ROAS / Rendimiento):** Relación entre los ingresos generados frente a lo invertido en publicidad.

In [12]:
# 1. Agregamos las métricas de marketing agrupadas por producto
marketing_por_producto = (
    df_marketing_limpio
    .groupby('producto')
    .agg(
        costo_marketing_total=('costo', 'sum'),
        canales_utilizados=('canal', lambda x: ", ".join(sorted(x.unique()))),
        cantidad_campanhas=('id_campanha', 'count')
    )
    .reset_index()
)

# 2. Reutilizamos el rendimiento de ventas por producto (de la Actividad 2)
ventas_por_producto = (
    df_ventas_limpio
    .groupby('producto')
    .agg(
        ingreso_generado=('ingreso_total', 'sum'),
        unidades_vendidas=('cantidad', 'sum')
    )
    .reset_index()
)

# 3. Integración mediante MERGE sobre la clave común 'producto'
df_integrado = pd.merge(
    ventas_por_producto,
    marketing_por_producto,
    on='producto',
    how='inner'
)

# 4. Cálculo de métrica de eficiencia publicitaria (Ratio Ingreso / Costo Publicitario)
df_integrado['relacion_ingreso_costo'] = df_integrado['ingreso_generado'] / df_integrado['costo_marketing_total']

# Ordenar por los productos más rentables en relación a su inversión publicitaria
df_integrado.sort_values(by='relacion_ingreso_costo', ascending=False, inplace=True)

# 5. Vista ejecutiva formateada
vista_integrada = df_integrado.copy()
vista_integrada['ingreso_generado'] = vista_integrada['ingreso_generado'].apply(
    lambda x: f"$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
vista_integrada['costo_marketing_total'] = vista_integrada['costo_marketing_total'].apply(
    lambda x: f"$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
vista_integrada['relacion_ingreso_costo'] = vista_integrada['relacion_ingreso_costo'].apply(
    lambda x: f"{x:,.2f}x".replace(".", ",")
)

print("=" * 85)
print("    TABLA INTEGRADA: RENDIMIENTO COMERCIAL vs. INVERSIÓN EN MARKETING")
print("=" * 85)
display(vista_integrada.head(10))

    TABLA INTEGRADA: RENDIMIENTO COMERCIAL vs. INVERSIÓN EN MARKETING


,producto,ingreso_generado,unidades_vendidas,costo_marketing_total,canales_utilizados,cantidad_campanhas,relacion_ingreso_costo
19,Lámpara de mesa,"$ 82.276,38",1112,"$ 15,93","Email, RRSS, TV",3,"5,164,87x"
20,Microondas,"$ 72.562,89",912,"$ 14,19","Email, RRSS, TV",3,"5,113,66x"
3,Auriculares,"$ 74.175,58",958,"$ 15,24","Email, RRSS, TV",3,"4,867,16x"
18,Lavadora,"$ 48.946,44",671,"$ 11,85","Email, RRSS, TV",3,"4,130,50x"
23,Proyector,"$ 48.187,00",626,"$ 12,47","Email, RRSS, TV",3,"3,864,23x"
16,Jarrón decorativo,"$ 51.130,88",672,"$ 13,27","Email, RRSS, TV",3,"3,853,12x"
13,Freidora eléctrica,"$ 50.155,15",630,"$ 13,56","Email, RRSS, TV",3,"3,698,76x"
7,Consola de videojuegos,"$ 46.174,41",623,"$ 12,77","Email, RRSS, TV",3,"3,615,85x"
15,Horno eléctrico,"$ 49.913,90",633,"$ 13,89","Email, RRSS, TV",3,"3,593,51x"
29,Televisor,"$ 47.038,04",632,"$ 13,13","Email, RRSS, TV",3,"3,582,49x"


### Hallazgos de la Integración (Ventas + Marketing):
* **Cobertura Total:** Los 30 productos con ventas registradas tuvieron campañas publicitarias asociadas en marketing, permitiendo un cruce del 100% sin pérdida de entidades.
* **Diversificación de Canales:** Los productos con mejor retorno combinaron campañas multicanal (Email, RRSS y TV).
* **Eficiencia Publicitaria:** La columna `relacion_ingreso_costo` permite a la dirección del negocio identificar qué productos devuelven más ingresos por cada peso invertido en pauta.